# Phase 05 — Data Formatting & Prompt Template Design
## CodeMentor-LLM
Converting CodeAlpaca-20K dataset to Llama-3 chat template format
for supervised fine-tuning (SFT).

## Goal
- Study Llama-3 chat template
- Write format_prompt() function
- Apply to entire dataset
- Save formatted dataset to HuggingFace Hub

# Libraries

In [ ]:
!pip install -q transformers==4.49.0 datasets==3.3.2 pandas==2.2.3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 66.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 60.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.2.3 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.12.0 which is incompatible.


# Login to HuggingFace

In [ ]:
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)
print("Logged in successfully")

Logged in successfully


# Load dataset from HuggingFace Hub

In [ ]:
from datasets import load_dataset

# Load primary dataset from HF Hub
print("Loading primary dataset...")
dataset = load_dataset("sahil2801/CodeAlpaca-20k")
print(dataset)
print(f"\nTotal samples: {len(dataset['train'])}")

Loading primary dataset...


README.md:   0%|          | 0.00/147 [00:00<?, ?B/s]

code_alpaca_20k.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/20022 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['output', 'instruction', 'input'],
        num_rows: 20022
    })
})

Total samples: 20022


# Study Llama-3 chat template

In [ ]:
from transformers import AutoTokenizer

# Load Llama-3 tokenizer
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct")

# Study the chat template with a sample message
messages = [
    {"role": "system", "content": "You are a helpful coding assistant."},
    {"role": "user", "content": "Write a Python function to reverse a string."},
    {"role": "assistant", "content": "Here is a Python function to reverse a string:\n```python\ndef reverse_string(s):\n    return s[::-1]\n```"}
]

# Apply chat template
formatted = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=False
)

print("Llama-3 Chat Template Format:")
print("=" * 50)
print(formatted)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

Llama-3 Chat Template Format:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a helpful coding assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>

Write a Python function to reverse a string.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Here is a Python function to reverse a string:
```python
def reverse_string(s):
    return s[::-1]
```<|eot_id|>


# Write format_prompt() function

In [ ]:
SYSTEM_PROMPT = "You are a helpful coding assistant. Answer coding questions clearly and concisely with working code examples."

def format_prompt(sample):
    """
    Converts a CodeAlpaca sample to Llama-3 chat template format.
    Args:
        sample: dict with keys instruction, input, output
    Returns:
        dict with key text containing formatted prompt
    """
    # Combine instruction and input if input exists
    if sample["input"] and sample["input"].strip():
        user_message = f"{sample['instruction']}\n\nInput:\n{sample['input']}"
    else:
        user_message = sample["instruction"]

    # Build messages
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message},
        {"role": "assistant", "content": sample["output"]}
    ]

    # Apply Llama-3 chat template
    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": formatted}

print("format_prompt() function defined successfully")

format_prompt() function defined successfully


# Test format_prompt() on single sample

In [ ]:
# Test on first sample
sample = dataset["train"][0]
print("ORIGINAL SAMPLE:")
print(f"Instruction: {sample['instruction']}")
print(f"Input: {sample['input']}")
print(f"Output: {sample['output']}")

print("\n" + "="*50)
print("FORMATTED SAMPLE:")
print("="*50)
formatted_sample = format_prompt(sample)
print(formatted_sample["text"])

ORIGINAL SAMPLE:
Instruction: Create an array of length 5 which contains all even numbers between 1 and 10.
Input: 
Output: arr = [2, 4, 6, 8, 10]

FORMATTED SAMPLE:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a helpful coding assistant. Answer coding questions clearly and concisely with working code examples.<|eot_id|><|start_header_id|>user<|end_header_id|>

Create an array of length 5 which contains all even numbers between 1 and 10.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

arr = [2, 4, 6, 8, 10]<|eot_id|>


# Test with sample that has input field

In [ ]:
# Find a sample that has input field
for i, sample in enumerate(dataset["train"]):
    if sample["input"] and sample["input"].strip():
        print(f"Sample index: {i}")
        print(f"Instruction: {sample['instruction']}")
        print(f"Input: {sample['input']}")
        print(f"Output: {sample['output']}")
        print("\n" + "="*50)
        print("FORMATTED:")
        print("="*50)
        formatted = format_prompt(sample)
        print(formatted["text"])
        break

Sample index: 2
Instruction: Write a replace method for a string class which replaces the given string with a given set of characters.
Input: string = "Hello World!"
replace_with = "Greetings!"
Output: def replace(self, replace_with):
    new_string = ""
    for char in self:
        if char == " ":
            new_string += replace_with
        else:
            new_string += char
    return new_string

FORMATTED:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a helpful coding assistant. Answer coding questions clearly and concisely with working code examples.<|eot_id|><|start_header_id|>user<|end_header_id|>

Write a replace method for a string class which replaces the given string with a given set of characters.

Input:
string = "Hello World!"
replace_with = "Greetings!"<|eot_id|><|start_header_id|>assistant<|end_header_id|>

def replace(self, replace_with):
    new_string = ""
    for char in self:
        if char == " ":
            new_string += replace_with

# Apply format_prompt() to entire dataset

In [ ]:
# Apply to entire dataset
print("Formatting entire dataset...")
formatted_dataset = dataset["train"].map(
    format_prompt,
    remove_columns=dataset["train"].column_names
)

print(f"Formatted dataset size: {len(formatted_dataset)}")
print(f"Columns: {formatted_dataset.column_names}")
print(f"\nFirst formatted sample:")
print(formatted_dataset[0]["text"][:200])

Formatting entire dataset...


Map:   0%|          | 0/20022 [00:00<?, ? examples/s]

Formatted dataset size: 20022
Columns: ['text']

First formatted sample:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a helpful coding assistant. Answer coding questions clearly and concisely with working code examples.<|eot_id|><|start_header_id|>u


# Check token lengths

In [ ]:
# Check token lengths of formatted samples
print("Checking token lengths...")

token_lengths = []
for sample in formatted_dataset:
    tokens = tokenizer(sample["text"], return_tensors="pt")
    token_lengths.append(tokens["input_ids"].shape[1])

import pandas as pd
lengths_series = pd.Series(token_lengths)

print(f"Min tokens  : {lengths_series.min()}")
print(f"Max tokens  : {lengths_series.max()}")
print(f"Mean tokens : {lengths_series.mean():.2f}")
print(f"Median      : {lengths_series.median()}")
print(f"\nSamples under 2048 tokens : {(lengths_series <= 2048).sum()}")
print(f"Samples over 2048 tokens  : {(lengths_series > 2048).sum()}")

Checking token lengths...
Min tokens  : 46
Max tokens  : 1174
Mean tokens : 116.79
Median      : 102.0

Samples under 2048 tokens : 20022
Samples over 2048 tokens  : 0


# Push formatted dataset to HuggingFace Hub

In [ ]:
from datasets import Dataset

# Convert to Dataset object
formatted_dataset_hf = Dataset.from_dict({"text": formatted_dataset["text"]})

# Push to HF Hub
print("Pushing formatted dataset to HF Hub...")
formatted_dataset_hf.push_to_hub("Abdulmoiz123/codementor-llm-formatted")
print("Formatted dataset pushed successfully")
print(f"Total samples: {len(formatted_dataset_hf)}")

Pushing formatted dataset to HF Hub...


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/21 [00:00<?, ?ba/s]

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


Formatted dataset pushed successfully
Total samples: 20022


# Formatted data

In [ ]:
# Save formatted dataset as JSONL for download
import json

# Save formatted dataset
with open("formatted_dataset.jsonl", "w") as f:
    for sample in formatted_dataset_hf:
        f.write(json.dumps(sample) + "\n")

print(f"Saved {len(formatted_dataset_hf)} samples to formatted_dataset.jsonl")

Saved 20022 samples to formatted_dataset.jsonl


# Raw data

In [ ]:
import json

# Save primary raw dataset
with open("codealapaca_20k_raw.jsonl", "w") as f:
    for sample in dataset["train"]:
        f.write(json.dumps(sample) + "\n")

print(f"Saved {len(dataset['train'])} samples to codealapaca_20k_raw.jsonl")

Saved 20022 samples to codealapaca_20k_raw.jsonl


# Backup data

In [13]:
# Save backup raw dataset
from datasets import load_dataset

backup_dataset = load_dataset("iamtarun/python_code_instructions_18k_alpaca")

with open("python_code_instructions_18k_raw.jsonl", "w") as f:
    for sample in backup_dataset["train"]:
        f.write(json.dumps(sample) + "\n")

print(f"Saved {len(backup_dataset['train'])} samples to python_code_instructions_18k_raw.jsonl")

README.md:   0%|          | 0.00/905 [00:00<?, ?B/s]

data/train-00000-of-00001-8b6e212f3e1ece(…):   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/18612 [00:00<?, ? examples/s]

Saved 18612 samples to python_code_instructions_18k_raw.jsonl
